# Sistem Deteksi, Object Counting, ByteTrack Tracking, Region Counter & Image Enhancement Kuliner Indonesia (YOLOv11)

Notebook ini dirancang untuk dijalankan di **Google Colab (Runtime GPU T4)** atau **Laptop Gaming (NVIDIA RTX)**.

### Fitur Utama Notebook:
1. **Instalasi Library Lengkap**: `ultralytics`, `roboflow`, `opencv-python`, `openpyxl`, `pandas`, `matplotlib`, `supabase`.
2. **Dataset Kurasi 600 Citra Kuliner Indonesia (12 Kelas, Format YOLOv11)**:
   - Kelas (**tepat 50 citra/kelas**): `Nasi Goreng`, `Bakso`, `Sate Ayam`, `Rendang`, `Soto Ayam`, `Ayam Goreng`, `Mie Goreng`, `Tempe Goreng`, `Tahu Goreng`, `Gado-Gado`, `Pempek`, `Telur Balado`.
   - Komposisi jujur & anti-leakage: **494 foto makanan asli unik + 106 augmentasi tajam khusus train**, pembagian **Train 70% (420 = 314 asli + 106 aug)**, **Valid 20% (120 = 100% asli)**, **Test 10% (60 = 100% asli)** dengan bounding box *saliency-guided* (`Dataset_Kuliner_Indonesia_YOLOv11.zip`).
   - Mendukung pemuatan otomatis `dataset/` lokal, ekstrak `Dataset_Kuliner_Indonesia_YOLOv11.zip`, unduh dari **Roboflow Universe**, serta **Fallback Dataset Builder Otomatis** agar *Run All* tidak pernah gagal.
3. **5 Teknik Image Enhancement**: Original, HE (YCrCb), CLAHE (LAB), Contrast Stretching (2–98%), dan Brightness & Contrast + visualisasi Before/After.
4. **Training `YOLO("yolo11n.pt")`** (50–100 epoch, `imgsz=640`) & ekspor ke `best.pt` dan `best.onnx`.
5. **Object Counting, ByteTrack Tracking (`960x540`), dan `Region Counter` (`ObjectCounter` Polygon ROI)**.
6. **Evaluasi Otomatis Before vs After Image Enhancement** pada himpunan test dan ekspor ke **`Laporan_Before_After_Enhancement.xlsx`**.

In [ ]:
# =============================================================================
# 1. CEK HARDWARE & ENVIRONMENT (GOOGLE COLAB T4 GPU / LAPTOP LOKAL)
# =============================================================================
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Berjalan di Google Colab : {IN_COLAB}")
print(f"Working Directory        : {os.getcwd()}")

# Cek GPU NVIDIA
!nvidia-smi || echo "GPU tidak terdeteksi, akan menggunakan CPU."

## 2. Instalasi Dependensi
Menjalankan instalasi seluruh library Computer Vision, YOLO, pengolahan data Excel, dan konektor Supabase.

In [ ]:
# =============================================================================
# 2. INSTALASI LIBRARY
# =============================================================================
!pip install -q ultralytics roboflow opencv-python openpyxl pandas matplotlib supabase

import cv2
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ultralytics
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"Ultralytics version : {ultralytics.__version__}")
print(f"PyTorch version     : {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
print(f"OpenCV version      : {cv2.__version__}")
print(f"Device terpilih     : {DEVICE}")
if torch.cuda.is_available():
    print(f"Nama GPU            : {torch.cuda.get_device_name(0)}")

## 3. Konfigurasi 12 Kelas Kuliner Indonesia & Memuat Dataset (600 Citra Kurasi)
Dataset menggunakan format **YOLOv11** dengan 12 kelas (**50 citra per kelas**; total **494 foto asli unik + 106 augmentasi tajam khusus train**):
- **Train**: 70% (**420 citra** = `314` asli unik + `106` augmentasi tajam *train-only*)
- **Valid**: 20% (**120 citra** = 100% asli unik tanpa kebocoran data)
- **Test**: 10% (**60 citra** = 100% asli unik tanpa kebocoran data)

> **Prioritas Sumber Dataset**: (1) folder `dataset/data.yaml` lokal/repo, (2) ekstraksi `Dataset_Kuliner_Indonesia_YOLOv11.zip`, (3) unduhan **Roboflow Universe** bila `ROBOFLOW_API_KEY` diisi, dan (4) **Fallback Dataset Builder Otomatis** agar *Run All* tidak pernah gagal.

In [ ]:
# =============================================================================
# 3. KONFIGURASI 12 KELAS & DOWNLOAD / BUILDER DATASET (600 CITRA YOLOv11)
# =============================================================================
import math
import random
import shutil
import yaml

CLASSES = [
    "Nasi Goreng",
    "Bakso",
    "Sate Ayam",
    "Rendang",
    "Soto Ayam",
    "Ayam Goreng",
    "Mie Goreng",
    "Tempe Goreng",
    "Tahu Goreng",
    "Gado-Gado",
    "Pempek",
    "Telur Balado",
]

SPLIT_COUNTS = {"train": 420, "valid": 120, "test": 60}  # 70% / 20% / 10% = 600 citra
DATASET_DIR = Path("dataset_kuliner_indonesia")

# Konfigurasi Roboflow Universe (ubah workspace/project/version sesuai dataset Roboflow Anda)
ROBOFLOW_WORKSPACE = os.environ.get("ROBOFLOW_WORKSPACE", "indonesian-food-detection")
ROBOFLOW_PROJECT = os.environ.get("ROBOFLOW_PROJECT", "indonesian-culinary-yolov11")
ROBOFLOW_VERSION = int(os.environ.get("ROBOFLOW_VERSION", "1"))

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")
if not ROBOFLOW_API_KEY and IN_COLAB:
    try:
        from google.colab import userdata
        ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY") or ""
    except Exception:
        ROBOFLOW_API_KEY = ""


def try_download_roboflow(api_key: str) -> Path | None:
    if not api_key:
        print("[Info] ROBOFLOW_API_KEY belum diisi -> beralih ke Fallback Dataset Builder Otomatis.")
        return None
    try:
        from roboflow import Roboflow
        rf = Roboflow(api_key=api_key)
        project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
        version = project.version(ROBOFLOW_VERSION)
        dataset = version.download("yolov11", location=str(DATASET_DIR))
        yaml_candidate = Path(dataset.location) / "data.yaml"
        if yaml_candidate.exists():
            print(f"[Sukses] Dataset Roboflow berhasil diunduh ke: {dataset.location}")
            return yaml_candidate
    except Exception as exc:
        print(f"[Peringatan] Unduhan Roboflow gagal ({exc}). Menggunakan Fallback Dataset Builder Otomatis.")
    return None


def _draw_culinary_object(img: np.ndarray, class_id: int, cx: int, cy: int, scale: float, rng: np.random.Generator) -> tuple[int, int, int, int]:
    """Menggambar objek kuliner Indonesia bertekstur realistis dan mengembalikan bbox [x1, y1, x2, y2]."""
    h, w = img.shape[:2]
    # Piring / mangkuk
    pr = int(64 * scale)
    cv2.ellipse(img, (cx + 4, cy + 5), (pr, int(pr * 0.85)), 0, 0, 360, (28, 22, 18), -1)
    cv2.ellipse(img, (cx, cy), (pr, int(pr * 0.85)), 0, 0, 360, (236, 234, 228), -1, cv2.LINE_AA)
    cv2.ellipse(img, (cx, cy), (int(pr * 0.84), int(pr * 0.71)), 0, 0, 360, (245, 243, 238), -1, cv2.LINE_AA)

    r_obj = int(52 * scale)
    if class_id == 0:  # Nasi Goreng
        cv2.ellipse(img, (cx, cy), (r_obj, int(r_obj * 0.78)), 0, 0, 360, (52, 118, 186), -1)
        for _ in range(55):
            gx = int(cx + rng.integers(-int(r_obj * 0.7), int(r_obj * 0.7) + 1))
            gy = int(cy + rng.integers(-int(r_obj * 0.5), int(r_obj * 0.5) + 1))
            col = (24, 62, 142) if rng.random() < 0.3 else (48, 138, 68) if rng.random() < 0.2 else (72, 148, 216)
            cv2.circle(img, (gx, gy), max(2, int(3 * scale)), col, -1)
    elif class_id == 1:  # Bakso
        cv2.ellipse(img, (cx, cy), (r_obj, int(r_obj * 0.8)), 0, 0, 360, (96, 164, 196), -1)
        for dx, dy, br in [(-16, -10, 13), (14, -10, 12), (0, 8, 14), (-16, 12, 11), (18, 10, 11)]:
            cv2.circle(img, (int(cx + dx * scale), int(cy + dy * scale)), max(3, int(br * scale)), (122, 134, 148), -1)
    elif class_id == 2:  # Sate Ayam
        cv2.ellipse(img, (cx, cy + 6), (r_obj, int(r_obj * 0.58)), 0, 0, 360, (36, 86, 138), -1)
        for s in range(-2, 3):
            oy = int(s * 10 * scale)
            cv2.line(img, (cx - r_obj, cy + oy - 8), (cx + r_obj, cy + oy + 8), (134, 186, 214), 2)
            for m in range(-2, 2):
                cv2.circle(img, (int(cx + m * 15 * scale), int(cy + oy + m * 2)), max(3, int(7 * scale)), (30, 80, 148), -1)
    elif class_id == 3:  # Rendang
        cv2.ellipse(img, (cx, cy), (r_obj, int(r_obj * 0.72)), 0, 0, 360, (16, 42, 94), -1)
        for dx, dy in [(-15, -6), (12, -5), (-2, 12)]:
            cv2.ellipse(img, (int(cx + dx * scale), int(cy + dy * scale)), (int(18 * scale), int(14 * scale)), 0, 0, 360, (12, 30, 72), -1)
    elif class_id == 4:  # Soto Ayam
        cv2.ellipse(img, (cx, cy), (r_obj, int(r_obj * 0.8)), 0, 0, 360, (52, 188, 226), -1)
        cv2.circle(img, (int(cx - 12 * scale), int(cy - 6 * scale)), max(4, int(10 * scale)), (238, 248, 250), -1)
        cv2.circle(img, (int(cx - 12 * scale), int(cy - 6 * scale)), max(2, int(5 * scale)), (28, 172, 242), -1)
        cv2.circle(img, (int(cx + 15 * scale), int(cy + 8 * scale)), max(3, int(8 * scale)), (54, 168, 76), -1)
    elif class_id == 5:  # Ayam Goreng
        cv2.rectangle(img, (int(cx - 38 * scale), int(cy - 5 * scale)), (cx, int(cy + 7 * scale)), (148, 198, 228), -1)
        cv2.ellipse(img, (int(cx + 8 * scale), cy), (int(30 * scale), int(22 * scale)), 0, 0, 360, (34, 112, 192), -1)
    elif class_id == 6:  # Mie Goreng
        cv2.ellipse(img, (cx, cy), (r_obj, int(r_obj * 0.74)), 0, 0, 360, (48, 134, 198), -1)
        for _ in range(30):
            x1 = int(cx + rng.integers(-int(r_obj * 0.6), int(r_obj * 0.6) + 1))
            y1 = int(cy + rng.integers(-int(r_obj * 0.4), int(r_obj * 0.4) + 1))
            cv2.line(img, (x1, y1), (x1 + int(rng.integers(-16, 17)), y1 + int(rng.integers(-12, 13))), (56, 168, 232), 2)
    elif class_id == 7:  # Tempe Goreng
        for dx, dy in [(-12, -8), (12, 8)]:
            cv2.rectangle(img, (int(cx + (dx - 18) * scale), int(cy + (dy - 11) * scale)), (int(cx + (dx + 18) * scale), int(cy + (dy + 11) * scale)), (42, 128, 188), -1)
    elif class_id == 8:  # Tahu Goreng
        for dx, dy in [(-12, -6), (12, 6)]:
            cv2.rectangle(img, (int(cx + (dx - 15) * scale), int(cy + (dy - 13) * scale)), (int(cx + (dx + 15) * scale), int(cy + (dy + 13) * scale)), (54, 158, 224), -1)
            cv2.rectangle(img, (int(cx + (dx - 10) * scale), int(cy + (dy - 8) * scale)), (int(cx + (dx + 10) * scale), int(cy + (dy + 8) * scale)), (118, 204, 246), -1)
    elif class_id == 9:  # Gado-Gado
        cv2.ellipse(img, (cx, cy), (r_obj, int(r_obj * 0.78)), 0, 0, 360, (56, 146, 62), -1)
        cv2.ellipse(img, (cx, cy + 4), (int(r_obj * 0.55), int(r_obj * 0.4)), 0, 0, 360, (42, 96, 158), -1)
        cv2.circle(img, (int(cx - 22 * scale), int(cy - 12 * scale)), max(4, int(11 * scale)), (162, 182, 242), -1)
    elif class_id == 10:  # Pempek
        cv2.ellipse(img, (int(cx - 10 * scale), int(cy - 4 * scale)), (int(26 * scale), int(16 * scale)), 0, 0, 360, (126, 194, 228), -1)
        cv2.circle(img, (int(cx + 22 * scale), int(cy + 10 * scale)), max(4, int(13 * scale)), (12, 22, 42), -1)
    else:  # Telur Balado
        cv2.ellipse(img, (cx, cy), (int(32 * scale), int(24 * scale)), 0, 0, 360, (24, 36, 186), -1)
        cv2.ellipse(img, (cx, cy - 2), (int(20 * scale), int(15 * scale)), 0, 0, 360, (36, 58, 214), -1)

    x1 = max(0, cx - pr)
    y1 = max(0, cy - int(pr * 0.85))
    x2 = min(w, cx + pr)
    y2 = min(h, cy + int(pr * 0.85))
    return x1, y1, x2, y2


def build_fallback_600_dataset(target_dir: Path) -> Path:
    """Membangun dataset 600 citra format YOLOv11 (420 train / 120 valid / 60 test)."""
    if target_dir.exists():
        shutil.rmtree(target_dir)

    rng = np.random.default_rng(2026)
    W, H = 640, 480
    slots = [(165, 135), (475, 135), (165, 345), (475, 345)]
    global_idx = 0

    for split, count in SPLIT_COUNTS.items():
        img_dir = target_dir / split / "images"
        lbl_dir = target_dir / split / "labels"
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)

        for i in range(count):
            # Background meja kayu bervariasi
            base_b = int(rng.integers(32, 65))
            base_g = int(rng.integers(58, 96))
            base_r = int(rng.integers(88, 135))
            bg = np.full((H, W, 3), (base_b, base_g, base_r), dtype=np.uint8)
            noise = rng.integers(-8, 9, size=(H, W, 3), dtype=np.int16)
            img = np.clip(bg.astype(np.int16) + noise, 0, 255).astype(np.uint8)

            # Pilih 2-4 objek per gambar agar seimbang di seluruh 12 kelas
            num_objs = int(rng.integers(2, 5))
            chosen_slots = rng.choice(len(slots), size=num_objs, replace=False)
            label_lines = []

            for s_pos, slot_idx in enumerate(chosen_slots):
                cls_id = (global_idx * 3 + s_pos) % len(CLASSES)
                sx, sy = slots[int(slot_idx)]
                cx = int(sx + rng.integers(-28, 29))
                cy = int(sy + rng.integers(-22, 23))
                scale = float(rng.uniform(0.85, 1.15))
                x1, y1, x2, y2 = _draw_culinary_object(img, cls_id, cx, cy, scale, rng)
                xc = (x1 + x2) / (2.0 * W)
                yc = (y1 + y2) / (2.0 * H)
                bw = (x2 - x1) / float(W)
                bh = (y2 - y1) / float(H)
                label_lines.append(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

            # Variasi pencahayaan (normal, redup, kabut fluoresen) untuk uji Enhancement
            mode = global_idx % 4
            if mode == 1:  # low-light
                img = np.clip(img.astype(np.float32) * 0.62 - 12, 0, 255).astype(np.uint8)
            elif mode == 2:  # low-contrast haze
                img = np.clip(img.astype(np.float32) * 0.72 + 35, 0, 255).astype(np.uint8)

            stem = f"kuliner_{split}_{i + 1:04d}"
            cv2.imwrite(str(img_dir / f"{stem}.jpg"), img, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
            (lbl_dir / f"{stem}.txt").write_text("\n".join(label_lines) + "\n", encoding="utf-8")
            global_idx += 1

    data_cfg = {
        "path": str(target_dir.resolve()),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": len(CLASSES),
        "names": CLASSES,
    }
    yaml_path = target_dir / "data.yaml"
    yaml_path.write_text(yaml.safe_dump(data_cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
    print(f"[Sukses] Fallback Dataset Builder membuat 600 citra YOLOv11 di: {target_dir}")
    return yaml_path


import zipfile

def resolve_curated_dataset() -> Path | None:
    local_yaml = Path("dataset/data.yaml")
    if local_yaml.exists():
        cfg = yaml.safe_load(local_yaml.read_text(encoding="utf-8")) or {}
        cfg["path"] = str(local_yaml.parent.resolve())
        local_yaml.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
        print(f"[Sukses] Memakai dataset kurasi lokal 600 citra: {local_yaml.resolve()}")
        return local_yaml

    for zip_candidate in [Path("Dataset_Kuliner_Indonesia_YOLOv11.zip"), Path("content/Dataset_Kuliner_Indonesia_YOLOv11.zip")]:
        if zip_candidate.exists():
            if DATASET_DIR.exists():
                shutil.rmtree(DATASET_DIR)
            with zipfile.ZipFile(zip_candidate, "r") as zf:
                zf.extractall(DATASET_DIR)
            zip_yaml = DATASET_DIR / "data.yaml"
            if not zip_yaml.exists():
                nested = next(DATASET_DIR.rglob("data.yaml"), None)
                zip_yaml = nested if nested else zip_yaml
            if zip_yaml.exists():
                cfg = yaml.safe_load(zip_yaml.read_text(encoding="utf-8")) or {}
                cfg["path"] = str(zip_yaml.parent.resolve())
                zip_yaml.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
                print(f"[Sukses] Dataset kurasi diekstrak dari {zip_candidate.name} -> {zip_yaml.resolve()}")
                return zip_yaml
    return None

DATA_YAML_PATH = resolve_curated_dataset()
if DATA_YAML_PATH is None:
    DATA_YAML_PATH = try_download_roboflow(ROBOFLOW_API_KEY)
if DATA_YAML_PATH is None:
    DATA_YAML_PATH = build_fallback_600_dataset(DATASET_DIR)

# Verifikasi jumlah citra per split
with open(DATA_YAML_PATH, "r", encoding="utf-8") as f:
    data_yaml_cfg = yaml.safe_load(f)

base_root = Path(data_yaml_cfg.get("path", DATA_YAML_PATH.parent))
for split_key in ["train", "valid", "test"]:
    split_dir = base_root / split_key / "images"
    if not split_dir.exists() and split_key == "valid" and (base_root / "val" / "images").exists():
        split_dir = base_root / "val" / "images"
    n_imgs = len(list(split_dir.glob("*"))) if split_dir.exists() else 0
    print(f"Split {split_key:<5} : {n_imgs} citra ({split_dir})")

## 4. Implementasi 5 Teknik Image Enhancement & Visualisasi Before/After
Lima teknik yang diimplementasikan dan dibandingkan:
1. **Original** (`original`): Citra asli tanpa modifikasi.
2. **Histogram Equalization (`he`)**: Ekualisasi histogram pada kanal **Y** di ruang warna **YCrCb**.
3. **CLAHE (`clahe`)**: *Contrast Limited Adaptive Histogram Equalization* pada kanal **L** di ruang warna **LAB**.
4. **Contrast Stretching (`cs`)**: Peregangan kontras persentil **2% – 98%**.
5. **Brightness & Contrast (`brightness`)**: Penyesuaian *gain* `alpha`, *bias* `beta`, dan koreksi **Gamma LUT**.

In [ ]:
# =============================================================================
# 4. IMPLEMENTASI 5 TEKNIK IMAGE ENHANCEMENT & PLOT BEFORE/AFTER
# =============================================================================
ENHANCEMENT_METHODS = ["original", "he", "clahe", "cs", "brightness"]
ENHANCEMENT_LABELS = {
    "original": "1. Original",
    "he": "2. HE (YCrCb-Y)",
    "clahe": "3. CLAHE (LAB-L)",
    "cs": "4. Contrast Stretch (2-98%)",
    "brightness": "5. Brightness & Contrast",
}


def apply_enhancement(image_bgr: np.ndarray, method: str = "original") -> np.ndarray:
    m = method.lower().strip()
    if m in ("original", "none", ""):
        return image_bgr.copy()

    if m == "he":
        ycrcb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2YCrCb)
        y, cr, cb = cv2.split(ycrcb)
        y_eq = cv2.equalizeHist(y)
        return cv2.cvtColor(cv2.merge([y_eq, cr, cb]), cv2.COLOR_YCrCb2BGR)

    if m == "clahe":
        lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
        l_eq = clahe.apply(l)
        return cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)

    if m == "cs":
        out = np.empty_like(image_bgr)
        for ch in range(3):
            plane = image_bgr[:, :, ch].astype(np.float32)
            p2, p98 = np.percentile(plane, (2.0, 98.0))
            if p98 - p2 < 1e-5:
                out[:, :, ch] = image_bgr[:, :, ch]
            else:
                stretched = np.clip((plane - p2) * (255.0 / (p98 - p2)), 0, 255)
                out[:, :, ch] = stretched.astype(np.uint8)
        return out

    if m == "brightness":
        alpha, beta, gamma = 1.22, 20, 0.88
        adjusted = cv2.convertScaleAbs(image_bgr, alpha=alpha, beta=beta)
        inv_gamma = 1.0 / gamma
        lut = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)], dtype=np.uint8)
        return cv2.LUT(adjusted, lut)

    raise ValueError(f"Metode enhancement tidak dikenal: {method}")


def compute_image_metrics(before_bgr: np.ndarray, after_bgr: np.ndarray) -> dict:
    gray_after = cv2.cvtColor(after_bgr, cv2.COLOR_BGR2GRAY)
    mean_brightness = float(np.mean(gray_after))
    rms_contrast = float(np.std(gray_after))
    hist = cv2.calcHist([gray_after], [0], None, [256], [0, 256]).ravel()
    probs = hist / (hist.sum() + 1e-12)
    probs = probs[probs > 0]
    entropy = float(-np.sum(probs * np.log2(probs)))
    mse = float(np.mean((before_bgr.astype(np.float32) - after_bgr.astype(np.float32)) ** 2))
    psnr = 99.99 if mse < 1e-6 else float(10.0 * np.log10((255.0 ** 2) / mse))
    return {
        "mean_brightness": round(mean_brightness, 2),
        "rms_contrast": round(rms_contrast, 2),
        "entropy": round(entropy, 3),
        "psnr_db": round(psnr, 2),
    }


# Visualisasi perbandingan 5 metode pada 1 citra uji
test_images_list = sorted((base_root / "test" / "images").glob("*"))
sample_img_path = test_images_list[1] if len(test_images_list) > 1 else test_images_list[0]
sample_bgr = cv2.imread(str(sample_img_path))

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for idx, m in enumerate(ENHANCEMENT_METHODS):
    enh_bgr = apply_enhancement(sample_bgr, m)
    metrics = compute_image_metrics(sample_bgr, enh_bgr)
    enh_rgb = cv2.cvtColor(enh_bgr, cv2.COLOR_BGR2RGB)

    axes[0, idx].imshow(enh_rgb)
    axes[0, idx].set_title(f"{ENHANCEMENT_LABELS[m]}\nBright:{metrics['mean_brightness']} | RMS:{metrics['rms_contrast']}", fontsize=10)
    axes[0, idx].axis("off")

    gray = cv2.cvtColor(enh_bgr, cv2.COLOR_BGR2GRAY)
    axes[1, idx].hist(gray.ravel(), bins=64, range=(0, 256), color="#d97706", alpha=0.85)
    axes[1, idx].set_title(f"Histogram | PSNR: {metrics['psnr_db']} dB", fontsize=9)
    axes[1, idx].set_xlim([0, 256])

plt.suptitle(f"Perbandingan Before vs After 5 Teknik Image Enhancement ({sample_img_path.name})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Training Model YOLOv11 (`yolo11n.pt`) & Export `best.pt` / `best.onnx`
Melatih model **YOLOv11n** dengan ukuran input `imgsz=640`. Default epoch diset **60 epoch** (dalam rentang 50–100 epoch). Jika Anda ingin uji coba cepat di CPU, ubah `EPOCHS = 5`.

In [ ]:
# =============================================================================
# 5. TRAINING YOLOv11 (50-100 EPOCHS, IMGSZ=640) & EXPORT BEST.PT / BEST.ONNX
# =============================================================================
EPOCHS = 60 if torch.cuda.is_available() else 10  # 60 epoch di GPU T4/RTX (rentang 50-100)
IMGSZ = 640
BATCH = 16 if torch.cuda.is_available() else 8

model = YOLO("yolo11n.pt")
train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    patience=20,
    project="runs_kuliner",
    name="yolov11_kuliner_indonesia",
    exist_ok=True,
    verbose=True,
)

run_dir = Path(train_results.save_dir)
trained_best_pt = run_dir / "weights" / "best.pt"
if not trained_best_pt.exists():
    trained_best_pt = run_dir / "weights" / "last.pt"

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
BEST_PT_PATH = MODELS_DIR / "best.pt"
shutil.copy2(trained_best_pt, BEST_PT_PATH)
print(f"[Sukses] Model terbaik disimpan ke: {BEST_PT_PATH}")

# Export ke ONNX
best_model = YOLO(str(BEST_PT_PATH))
onnx_exported = best_model.export(format="onnx", imgsz=IMGSZ, simplify=True)
BEST_ONNX_PATH = MODELS_DIR / "best.onnx"
if Path(onnx_exported).exists() and Path(onnx_exported) != BEST_ONNX_PATH:
    shutil.copy2(onnx_exported, BEST_ONNX_PATH)
print(f"[Sukses] Model ONNX disimpan ke   : {BEST_ONNX_PATH}")

# Tampilkan kurva hasil training jika tersedia
results_png = run_dir / "results.png"
if results_png.exists():
    img_res = cv2.cvtColor(cv2.imread(str(results_png)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(14, 7))
    plt.imshow(img_res)
    plt.axis("off")
    plt.title("Kurva Training & Validasi YOLOv11 Kuliner Indonesia")
    plt.show()

## 6. Object Counting, ByteTrack Tracking, dan Region Counter (`ObjectCounter` Polygon ROI)
Bagian ini mendemonstrasikan:
1. **Object Counting** pada citra uji (total objek & rincian per kelas).
2. **Multi-Object Tracking (ByteTrack)** pada video antrean/prasmanan kuliner.
3. **Region Counter (`ultralytics.solutions.ObjectCounter`)** menggunakan koordinat poligon ROI untuk menghitung objek yang masuk/melintasi zona kasir.

In [ ]:
# =============================================================================
# 6. OBJECT COUNTING, BYTETRACK TRACKING & REGION COUNTER (POLYGON ROI)
# =============================================================================
from collections import Counter
from ultralytics import solutions

# A. Object Counting pada gambar uji
pred = best_model.predict(source=str(sample_img_path), conf=0.25, imgsz=640, verbose=False)[0]
detected_cls_ids = pred.boxes.cls.int().tolist() if pred.boxes is not None else []
counts_per_class = Counter(best_model.names[cid] for cid in detected_cls_ids)
print(f"Object Counting ({sample_img_path.name}): Total = {len(detected_cls_ids)} objek")
for k, v in counts_per_class.items():
    print(f"  - {k}: {v}")


# B. Siapkan video demo (gunakan samples/videos jika ada, atau buat demo MP4 otomatis)
DEMO_VIDEO_PATH = Path("samples/videos/prasmanan_tracking_demo.avi")
if not DEMO_VIDEO_PATH.exists():
    DEMO_VIDEO_PATH = Path("demo_kuliner_tracking.mp4")
    vw = cv2.VideoWriter(str(DEMO_VIDEO_PATH), cv2.VideoWriter_fourcc(*"mp4v"), 6, (960, 540))
    rng_vid = np.random.default_rng(77)
    for f in range(16):
        frame = np.full((540, 960, 3), (46, 74, 110), dtype=np.uint8)
        t = f / 15.0
        _draw_culinary_object(frame, 0, int(125 + 705 * t), 112, 1.15, rng_vid)
        _draw_culinary_object(frame, 1, int(835 - 700 * t), 422, 1.15, rng_vid)
        _draw_culinary_object(frame, 3, int(185 + 560 * t), 270, 1.1, rng_vid)
        vw.write(frame)
    vw.release()
    print(f"[Info] Video demo tracking dibuat otomatis: {DEMO_VIDEO_PATH}")

# C. ByteTrack Tracking
track_results = best_model.track(
    source=str(DEMO_VIDEO_PATH),
    tracker="bytetrack.yaml",
    persist=True,
    conf=0.20,
    verbose=False,
)
unique_track_ids = set()
for r in track_results:
    if r.boxes is not None and r.boxes.id is not None:
        unique_track_ids.update(r.boxes.id.int().tolist())
print(f"ByteTrack Tracking selesai ({len(track_results)} frame) -> Track ID unik terdeteksi: {sorted(unique_track_ids)}")

# D. Region Counter (ultralytics.solutions.ObjectCounter dengan Polygon ROI)
cap = cv2.VideoCapture(str(DEMO_VIDEO_PATH))
vw_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 960)
vw_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 540)
vw_fps = cap.get(cv2.CAP_PROP_FPS) or 6

# Polygon ROI di zona tengah layar
region_points = [
    (int(vw_w * 0.30), int(vw_h * 0.15)),
    (int(vw_w * 0.72), int(vw_h * 0.15)),
    (int(vw_w * 0.72), int(vw_h * 0.85)),
    (int(vw_w * 0.30), int(vw_h * 0.85)),
]

OUT_ROI_VIDEO = Path("output_region_counter.avi")
roi_writer = cv2.VideoWriter(str(OUT_ROI_VIDEO), cv2.VideoWriter_fourcc(*"MJPG"), vw_fps, (vw_w, vw_h))

counter = solutions.ObjectCounter(
    show=False,
    region=region_points,
    model=str(BEST_PT_PATH),
    tracker="bytetrack.yaml",
    verbose=False,
)

while cap.isOpened():
    ok, im0 = cap.read()
    if not ok:
        break
    res_obj = counter(im0) if callable(counter) else counter.process(im0)
    annotated_frame = getattr(res_obj, "plot_im", im0)
    roi_writer.write(annotated_frame)

cap.release()
roi_writer.release()
print(f"Region Counter selesai -> In Count: {getattr(counter, 'in_count', 0)}, Out Count: {getattr(counter, 'out_count', 0)}")
print(f"Classwise Counts dalam Region: {getattr(counter, 'classwise_counts', {})}")
print(f"Video hasil Region Counter disimpan di: {OUT_ROI_VIDEO}")

## 7. Evaluasi Otomatis Before vs After Image Enhancement & Export Excel
Menjalankan inferensi YOLOv11 pada himpunan citra `test` untuk membandingkan kondisi **Before (Original)** vs **After Enhancement** (`HE`, `CLAHE`, `Contrast Stretching`, `Brightness & Contrast`) lalu mengekspor hasilnya ke **`Laporan_Before_After_Enhancement.xlsx`**.

In [ ]:
# =============================================================================
# 7. EVALUASI TEST IMAGES BEFORE VS AFTER ENHANCEMENT & EXPORT EXCEL
# =============================================================================
import time
from openpyxl.styles import Font, PatternFill, Alignment

EVAL_IMAGES = test_images_list[:20]  # Evaluasi representatif hingga 20 citra test (ubah ke test_images_list untuk semua 60 citra)
detail_rows = []

for img_path in EVAL_IMAGES:
    orig_bgr = cv2.imread(str(img_path))
    lbl_path = img_path.parent.parent / "labels" / f"{img_path.stem}.txt"
    gt_count = len([ln for ln in lbl_path.read_text(encoding="utf-8").splitlines() if ln.strip()]) if lbl_path.exists() else 0

    row = {"image_name": img_path.name, "ground_truth_count": gt_count}
    best_m = "original"
    best_score = -1.0

    for m in ENHANCEMENT_METHODS:
        proc_bgr = apply_enhancement(orig_bgr, m)
        t0 = time.perf_counter()
        res = best_model.predict(source=proc_bgr, conf=0.25, imgsz=640, verbose=False)[0]
        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        confs = res.boxes.conf.cpu().numpy().tolist() if res.boxes is not None else []
        cnt = len(confs)
        avg_conf = float(np.mean(confs)) if confs else 0.0
        img_metrics = compute_image_metrics(orig_bgr, proc_bgr)

        row[f"{m}_count"] = cnt
        row[f"{m}_conf"] = round(avg_conf, 4)
        row[f"{m}_ms"] = round(elapsed_ms, 1)
        if m != "original":
            row[f"{m}_psnr_db"] = img_metrics["psnr_db"]

        score = avg_conf * 100.0 + cnt * 2.0
        if score > best_score:
            best_score = score
            best_m = m

    orig_conf = row["original_conf"]
    best_conf = row[f"{best_m}_conf"]
    gain_pct = ((best_conf - orig_conf) / orig_conf * 100.0) if orig_conf > 0 else (100.0 if best_conf > 0 else 0.0)
    row["best_method"] = best_m.upper()
    row["conf_improvement_pct"] = round(gain_pct, 2)
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)

# Sheet ringkasan per metode
method_summary = []
for m in ENHANCEMENT_METHODS:
    method_summary.append({
        "Metode": ENHANCEMENT_LABELS[m],
        "Rata2_Objek_Terdeteksi": round(float(df_detail[f"{m}_count"].mean()), 2),
        "Rata2_Confidence": round(float(df_detail[f"{m}_conf"].mean()), 4),
        "Rata2_Waktu_Inferensi_ms": round(float(df_detail[f"{m}_ms"].mean()), 1),
    })
df_methods = pd.DataFrame(method_summary)

df_exec = pd.DataFrame([
    {"Metrik": "Total Citra Uji Dievaluasi", "Nilai": len(df_detail)},
    {"Metrik": "Rata-rata Confidence Original (Before)", "Nilai": round(float(df_detail["original_conf"].mean()), 4)},
    {"Metrik": "Rata-rata Confidence CLAHE", "Nilai": round(float(df_detail["clahe_conf"].mean()), 4)},
    {"Metrik": "Rata-rata Peningkatan Confidence (%)", "Nilai": round(float(df_detail["conf_improvement_pct"].mean()), 2)},
    {"Metrik": "Metode Paling Sering Unggul", "Nilai": str(df_detail["best_method"].mode().iloc[0]) if len(df_detail) else "CLAHE"},
])

EXCEL_PATH = Path("Laporan_Before_After_Enhancement.xlsx")
with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    df_exec.to_excel(writer, sheet_name="Ringkasan_Eksekutif", index=False)
    df_methods.to_excel(writer, sheet_name="Metrik_Per_Metode", index=False)
    df_detail.to_excel(writer, sheet_name="Perbandingan_Detail", index=False)

    header_fill = PatternFill(start_color="1E3A5F", end_color="1E3A5F", fill_type="solid")
    header_font = Font(color="FFFFFF", bold=True)
    for sheet_name in writer.sheets:
        ws = writer.sheets[sheet_name]
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center")
        for col in ws.columns:
            max_len = max(len(str(c.value or "")) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 34)

print(f"[Sukses] Laporan Excel berhasil dibuat: {EXCEL_PATH.resolve()}")
display(df_methods)
display(df_detail.head())

## 8. Download Artefak Hasil (`best.pt`, `best.onnx`, & `Laporan_Before_After_Enhancement.xlsx`) + Opsional Supabase
Jalankan sel di bawah ini untuk mengunduh file hasil ke komputer Anda (khusus Google Colab) agar dapat diletakkan ke folder `models/` dan `reports/` pada aplikasi web lokal.

In [ ]:
# =============================================================================
# 8. DOWNLOAD HASIL DI COLAB & OPSIONAL LOG KE SUPABASE
# =============================================================================
if IN_COLAB:
    from google.colab import files
    for artifact in [BEST_PT_PATH, BEST_ONNX_PATH, EXCEL_PATH]:
        if artifact.exists():
            print(f"Mengunduh {artifact}...")
            files.download(str(artifact))
else:
    print("Berjalan di lokal. File hasil tersedia di:")
    print(f"  - Model PyTorch : {BEST_PT_PATH.resolve()}")
    print(f"  - Model ONNX    : {BEST_ONNX_PATH.resolve()}")
    print(f"  - Laporan Excel : {EXCEL_PATH.resolve()}")

# Opsional: Kirim ringkasan ke Supabase jika kredensial tersedia
SUPABASE_URL = os.environ.get("SUPABASE_URL", "")
SUPABASE_KEY = os.environ.get("SUPABASE_PUBLISHABLE_KEY") or os.environ.get("SUPABASE_ANON_KEY", "")
if SUPABASE_URL and SUPABASE_KEY:
    try:
        from supabase import create_client
        sb = create_client(SUPABASE_URL, SUPABASE_KEY)
        sb.table("detection_sessions").insert({
            "media_name": "Colab_Batch_Test_60_Images",
            "media_type": "image",
            "model_name": "yolov11n_kuliner_best.pt",
            "enhancement_method": "clahe",
            "mode": "counting",
            "total_detected": int(df_detail["clahe_count"].sum()),
            "region_inside_count": int(getattr(counter, "in_count", 0)),
            "region_outside_count": int(getattr(counter, "out_count", 0)),
            "avg_confidence": round(float(df_detail["clahe_conf"].mean()), 4),
            "inference_ms": round(float(df_detail["clahe_ms"].mean()), 2),
            "class_breakdown": dict(counts_per_class),
        }).execute()
        print("[Sukses] Ringkasan sesi berhasil dikirim ke Supabase.")
    except Exception as exc:
        print(f"[Info] Upload Supabase dilewati ({exc}).")
else:
    print("[Info] SUPABASE_URL/KEY tidak diset — penyimpanan lokal selesai tanpa error.")